In [76]:
import pandas as pd
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack import Pipeline

In [77]:
dataset = pd.read_csv("/Users/archith25/Desktop/Data_PreProcessing/marine_ocr_output.csv")
docs1 = [
    {
        "content": f"{row['name']} - {row['significance']}. Coordinates: "
                   f"Latitudes: [{row['lat_1']}, {row['lat_2']}, {row['lat_3']}, {row['lat_4']}], "
                   f"Longitudes: [{row['lon_1']}, {row['lon_2']}, {row['lon_3']}, {row['lon_4']}]"
    }
    for index, row in dataset.iterrows()
]

docs = [Document(content=doc["content"]) for doc in docs1]

In [78]:
document_store = InMemoryDocumentStore()
document_store.write_documents(docs)

25

In [118]:
retriever_13 = InMemoryBM25Retriever(document_store)

In [119]:
template = """
Given the context,
Extract the following information from the input text:
* Location: [latitude, longitude]
* Time: [timestamp or time range]
* Object Type: [ship, aircraft, submarine, etc.]
* Object Description: [size, color, movement, speed, direction, etc.]
* Source: [radar, satellite, human observation]
* Confidence Level: [high, medium, low]
If possible cross-reference the reports with information from context (e.g. previous sightings or known entities).

context:
{% for document in documents %}
    {{ document.content }}
{% endfor %}
Input_text: {{input_text}}
Answer: 
"""

In [120]:
prompt_builder_13 = PromptBuilder(template=template)

generator_13 = OllamaGenerator(model = "mistral", 
                              url = "http://localhost:11434", 
                              generation_kwargs={"num_predict" : 100,
                                                 "temperature" : 0.7 
                                                }, 
                              timeout = 600
)

In [121]:
basic_rag_pipeline = Pipeline()
basic_rag_pipeline.add_component("retriever", retriever_13)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder_13)
basic_rag_pipeline.add_component("llm", generator_13)
basic_rag_pipeline.connect("retriever", "prompt_builder.documents")
basic_rag_pipeline.connect("prompt_builder", "llm")

🚅 Components
  - retriever: InMemoryBM25Retriever
  - prompt_builder: PromptBuilder
  - llm: OllamaGenerator
🛤️ Connections
  - retriever.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> llm.prompt (str)

In [122]:
Input_text = """
A distress signal was received from a fishing vessel, "Oceanic Hope", located at approximately 12.345° N, 78.910° E. 
The vessel is experiencing engine failure and is drifting towards a reef. The vessel has a crew of 10 and is carrying 
a cargo of fish. Urgent assistance is required to prevent a potential maritime disaster.
"""

response = basic_rag_pipeline.run(
    {
        "retriever" : {"query": Input_text},
        "prompt_builder" : {"input_text": Input_text}
    }
)

print(response["llm"]["replies"][0])

 * Location: [Latitude: 12.345° N, Longitude: 78.910° E]
* Time: Not provided in the text (could be assumed as current based on the distress signal context)
* Object Type: Fishing vessel ("Oceanic Hope")
* Object Description: Experiencing engine failure, drifting towards a reef, crew of 10, carrying cargo of fish.
* Source: Not
